In [1]:
import duckdb

con = duckdb.connect()

In [ ]:
def calculate_stats(paths):
    if isinstance(paths, str):
        paths = [paths]

    parquet_paths = ", ".join(f"'{path}'" for path in paths)

    # Get schema
    schema = con.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet([{parquet_paths}])
    """).pl()

    numeric_types = [
        "TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT",
        "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT", "UHUGEINT",
        "FLOAT", "DOUBLE", "DECIMAL"
    ]

    numeric_columns = schema.filter(
        schema["column_type"]
        .str.to_uppercase()
        .str.split("<")
        .list.first()
        .is_in(numeric_types)
    )["column_name"].to_list()

    aggregations = ",\n".join(
        f"""
        AVG("{col}") AS "{col}_mean",
        STDDEV_SAMP("{col}") AS "{col}_std",
        MIN("{col}") AS "{col}_min",
        MAX("{col}") AS "{col}_max"
        """
        for col in numeric_columns
    )

    return con.sql(f"""
        SELECT
            {aggregations}
        FROM read_parquet([{parquet_paths}])
    """).pl()

stats_cn = calculate_stats(
    "../data/raw/voxaging/data/cn/*.parquet"
)

stats_en = calculate_stats(
    "../data/raw/voxaging/data/en/*.parquet"
)

stats_all = calculate_stats([
    "../data/raw/voxaging/data/en/*.parquet",
    "../data/raw/voxaging/data/cn/*.parquet"
])

In [22]:
print("CN:")
display(stats_cn)
print("EN:")
display(stats_en)
print("All:")
display(stats_all)

CN:


week_id_mean,week_id_std,week_id_min,week_id_max,num_samples_mean,num_samples_std,num_samples_min,num_samples_max,duration_mean,duration_std,duration_min,duration_max
f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64
121.863563,92.158522,0,467,88299.291637,118371.902949,2880,960160,5.518706,7.398244,0.18,60.01


EN:


week_id_mean,week_id_std,week_id_min,week_id_max,num_samples_mean,num_samples_std,num_samples_min,num_samples_max,duration_mean,duration_std,duration_min,duration_max
f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64
186.306126,140.996537,0,815,126001.001561,164698.161383,2560,960160,7.875063,10.293635,0.16,60.01


All:


week_id_mean,week_id_std,week_id_min,week_id_max,num_samples_mean,num_samples_std,num_samples_min,num_samples_max,duration_mean,duration_std,duration_min,duration_max
f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64
171.265588,134.035085,0,815,117201.630462,155945.898378,2560,960160,7.325102,9.746619,0.16,60.01


In [39]:
col = "duration" # week_id / num_samples / duration
def missing_stats(path):
    return con.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (WHERE {col} IS NULL) AS {col}_missing,
            ROUND(
                100.0 * COUNT(*) FILTER (WHERE {col} IS NULL) / COUNT(*),
                2
            ) AS {col}_missing_pct
        FROM '{path}';
    """).pl()


stats_missing_en = missing_stats(
    "../data/raw/voxaging/data/en/*.parquet"
)

stats_missing_cn = missing_stats(
    "../data/raw/voxaging/data/cn/*.parquet"
)

stats_missing_all = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE {col} IS NULL) AS {col}_missing,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE {col} IS NULL) / COUNT(*),
            2
        ) AS {col}_missing_pct
    FROM read_parquet([
        '../data/raw/voxaging/data/en/*.parquet',
        '../data/raw/voxaging/data/cn/*.parquet'
    ]);
""").pl()

In [40]:
print("EN:")
display(stats_missing_en)
print("CN:")
display(stats_missing_cn)
print("All:")
display(stats_missing_all)

EN:


total_rows,duration_missing,duration_missing_pct
i64,i64,f64
274132,0,0.0


CN:


total_rows,duration_missing,duration_missing_pct
i64,i64,f64
83460,0,0.0


All:


total_rows,duration_missing,duration_missing_pct
i64,i64,f64
357592,0,0.0


In [23]:
schema_en = con.sql("""
    DESCRIBE
    SELECT *
    FROM '../data/raw/voxaging/data/en/*.parquet'
""").pl()

schema_cn = con.sql("""
    DESCRIBE
    SELECT *
    FROM '../data/raw/voxaging/data/cn/*.parquet'
""").pl()

schema_all = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet([
        '../data/raw/voxaging/data/en/*.parquet',
        '../data/raw/voxaging/data/cn/*.parquet'
    ])
""").pl()

In [27]:
display(
    schema_en.equals(schema_cn)
    and schema_cn.equals(schema_all)
)
print("All:")
display(schema_all)

True

All:


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""domain""","""VARCHAR""","""YES""",null,null,null
"""speaker_id""","""VARCHAR""","""YES""",null,null,null
"""week_id""","""BIGINT""","""YES""",null,null,null
"""utt_id""","""VARCHAR""","""YES""",null,null,null
"""audio""","""STRUCT(""array"" DOUBLE[], sampl…","""YES""",null,null,null
"""num_samples""","""BIGINT""","""YES""",null,null,null
"""duration""","""DOUBLE""","""YES""",null,null,null


In [7]:
con.sql("""
    SELECT COUNT(*)
    FROM '../data/raw/voxaging/data/en/*.parquet'
        """).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       274132 │
└──────────────┘



In [29]:
n_rows_en = con.sql("""
    SELECT COUNT(*) AS n_rows
    FROM '../data/raw/voxaging/data/en/*.parquet'
""").pl()

n_rows_cn = con.sql("""
    SELECT COUNT(*) AS n_rows
    FROM '../data/raw/voxaging/data/cn/*.parquet'
""").pl()

n_rows_all = con.sql("""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet([
        '../data/raw/voxaging/data/en/*.parquet',
        '../data/raw/voxaging/data/cn/*.parquet'
    ])
""").pl()

In [31]:
print("EN: {}".format(n_rows_en.item()))
print("CN: {}".format(n_rows_cn.item()))
print("All: {}".format(n_rows_all.item()))

EN: 274132
CN: 83460
All: 357592
